In [ ]:
# Install required packages
!pip install tldextract
!pip install tldextract imbalanced-learn tqdm

# Import libraries
import pandas as pd
import numpy as np
import tldextract
from urllib.parse import urlparse
import re
import math
import collections
from scipy.stats import entropy
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
# Step 2: Upload phishing dataset
print("Upload phishing dataset:")
uploaded_phish = files.upload()

# Load phishing dataset into pandas DataFrame
phish_df = pd.read_csv("online-valid_ds.csv")

# Show first 5 rows
print("Phishing dataset preview:")
print(phish_df.head())
print("\nPhishing dataset columns:", phish_df.columns.tolist())

Upload phishing dataset:


Saving online-valid_ds.csv to online-valid_ds.csv
Phishing dataset preview:
   phish_id                                                url  \
0   9222036               http://allegro.pl-oferta3353835.shop   
1   8336498  https://support-auth-network-login.firebaseapp...   
2   8441001  https://docs.google.com/presentation/d/e/2PACX...   
3   8684317  https://docs.google.com/presentation/d/e/2PACX...   
4   9213448  https://atendimentopersonalizadopj.online/aces...   

                                    phish_detail_url  \
0  http://www.phishtank.com/phish_detail.php?phis...   
1  http://www.phishtank.com/phish_detail.php?phis...   
2  http://www.phishtank.com/phish_detail.php?phis...   
3  http://www.phishtank.com/phish_detail.php?phis...   
4  http://www.phishtank.com/phish_detail.php?phis...   

             submission_time verified          verification_time online  \
0  2025-09-25T14:32:40+00:00      yes  2025-09-25T14:41:45+00:00    yes   
1  2023-10-18T22:05:32+00:00      yes  2

In [ ]:
# Step 3: Upload benign dataset
print("Upload benign dataset:")
uploaded_benign = files.upload()

# Load benign dataset into pandas DataFrame
benign_df = pd.read_csv("Benign_url_file.csv", header=None, names=["url"])

# Show first 5 rows
print("Benign dataset preview:")
print(benign_df.head())
print("\nBenign dataset columns:", benign_df.columns.tolist())

Upload benign dataset:


Saving Benign_url_file.csv to Benign_url_file.csv
Benign dataset preview:
                                                 url
0  http://1337x.to/torrent/1048648/American-Snipe...
1  http://1337x.to/torrent/1110018/Blackhat-2015-...
2  http://1337x.to/torrent/1122940/Blackhat-2015-...
3  http://1337x.to/torrent/1124395/Fast-and-Furio...
4  http://1337x.to/torrent/1145504/Avengers-Age-o...

Benign dataset columns: ['url']


In [ ]:
# Step 4: Add labels
phish_df['label'] = 1   # phishing = 1
benign_df['label'] = 0  # benign = 0

# Sample 20,000 rows from each dataset
phish_sampled = phish_df.sample(n=30000, random_state=42)
benign_sampled = benign_df.sample(n=30000, random_state=42)

print("Phishing sample shape:", phish_sampled.shape)
print("Benign sample shape:", benign_sampled.shape)

Phishing sample shape: (30000, 10)
Benign sample shape: (30000, 2)


In [ ]:
# Step 5: Combine 20k phishing + 20k benign samples
combined_df = pd.concat(
    [phish_sampled[['url', 'label']], benign_sampled[['url', 'label']]],
    ignore_index=True
)

print("Combined dataset shape:", combined_df.shape)
print("Label distribution:\n", combined_df['label'].value_counts())

# Step 6: Clean dataset
before_shape = combined_df.shape

# Drop rows with missing or empty URLs
combined_df = combined_df.dropna(subset=['url'])
combined_df = combined_df[combined_df['url'].str.strip() != ""]

# Drop duplicate URLs
combined_df = combined_df.drop_duplicates(subset=['url'], keep='first')

after_shape = combined_df.shape

print("Shape before cleaning:", before_shape)
print("Shape after cleaning:", after_shape)
print("Label distribution after cleaning:\n", combined_df['label'].value_counts())

Combined dataset shape: (60000, 2)
Label distribution:
 label
1    30000
0    30000
Name: count, dtype: int64
Shape before cleaning: (60000, 2)
Shape after cleaning: (59992, 2)
Label distribution after cleaning:
 label
0    30000
1    29992
Name: count, dtype: int64


In [ ]:
# =====================
# Helper: Entropy
# =====================
def shannon_entropy(s):
    """Compute Shannon entropy of a string."""
    if not s:
        return 0.0
    prob = [float(s.count(c)) / len(s) for c in dict.fromkeys(s)]
    return -sum([p * math.log(p, 2) for p in prob])

# =====================
# Main Feature Extractor
# =====================
def extract_url_features(url: str):
    """
    Extract features from a single URL — consistent with training version.
    Returns a dictionary (ready for DataFrame construction).
    """

    parsed = urlparse(url)
    domain = parsed.netloc or ""
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""
    subdomains = domain.split('.')[:-2] if len(domain.split('.')) > 2 else []
    sub_str = '.'.join(subdomains)

    features = {
        # --- URL-level ---
        "url_length": len(url),
        "number_of_dots_in_url": url.count('.'),
        "having_repeated_digits_in_url": int(any(url.count(d) > 1 for d in "0123456789")),
        "number_of_digits_in_url": sum(c.isdigit() for c in url),
        "number_of_special_char_in_url": sum(not c.isalnum() for c in url),
        "number_of_hyphens_in_url": url.count('-'),
        "number_of_underline_in_url": url.count('_'),
        "number_of_slash_in_url": url.count('/'),
        "number_of_questionmark_in_url": url.count('?'),
        "number_of_equal_in_url": url.count('='),
        "number_of_at_in_url": url.count('@'),
        "number_of_dollar_in_url": url.count('$'),
        "number_of_exclamation_in_url": url.count('!'),
        "number_of_hashtag_in_url": url.count('#'),
        "number_of_percent_in_url": url.count('%'),

        # --- Domain-level ---
        "domain_length": len(domain),
        "number_of_dots_in_domain": domain.count('.'),
        "number_of_hyphens_in_domain": domain.count('-'),
        "having_special_characters_in_domain": int(any(not c.isalnum() and c not in ".-" for c in domain)),
        "number_of_special_characters_in_domain": sum(not c.isalnum() and c not in ".-" for c in domain),
        "having_digits_in_domain": int(any(c.isdigit() for c in domain)),
        "number_of_digits_in_domain": sum(c.isdigit() for c in domain),
        "having_repeated_digits_in_domain": int(any(domain.count(d) > 1 for d in "0123456789")),
        "number_of_subdomains": len(subdomains),
        "having_dot_in_subdomain": int('.' in sub_str),
        "having_hyphen_in_subdomain": int('-' in sub_str),
        "average_subdomain_length": np.mean([len(s) for s in subdomains]) if subdomains else 0.0,
        "average_number_of_dots_in_subdomain": np.mean([s.count('.') for s in subdomains]) if subdomains else 0,
        "average_number_of_hyphens_in_subdomain": np.mean([s.count('-') for s in subdomains]) if subdomains else 0,
        "having_special_characters_in_subdomain": int(any(not c.isalnum() for c in sub_str)) if sub_str else 0,
        "number_of_special_characters_in_subdomain": sum(not c.isalnum() for c in sub_str),
        "having_digits_in_subdomain": int(any(c.isdigit() for c in sub_str)) if sub_str else 0,
        "number_of_digits_in_subdomain": sum(c.isdigit() for c in sub_str),
        "having_repeated_digits_in_subdomain": int(any(sub_str.count(d) > 1 for d in "0123456789")) if sub_str else 0,

        # --- Path / query / fragment ---
        "having_path": int(bool(path)),
        "path_length": len(path),
        "having_query": int(bool(query)),
        "having_fragment": int(bool(fragment)),
        "having_anchor": int('#' in url),

        # --- Entropy ---
        "entropy_of_url": shannon_entropy(url),
        "entropy_of_domain": shannon_entropy(domain)
    }

    # maintain same column order as used in training
    feature_order = [
        "url_length","number_of_dots_in_url","having_repeated_digits_in_url",
        "number_of_digits_in_url","number_of_special_char_in_url","number_of_hyphens_in_url",
        "number_of_underline_in_url","number_of_slash_in_url","number_of_questionmark_in_url",
        "number_of_equal_in_url","number_of_at_in_url","number_of_dollar_in_url",
        "number_of_exclamation_in_url","number_of_hashtag_in_url","number_of_percent_in_url",
        "domain_length","number_of_dots_in_domain","number_of_hyphens_in_domain",
        "having_special_characters_in_domain","number_of_special_characters_in_domain",
        "having_digits_in_domain","number_of_digits_in_domain","having_repeated_digits_in_domain",
        "number_of_subdomains","having_dot_in_subdomain","having_hyphen_in_subdomain",
        "average_subdomain_length","average_number_of_dots_in_subdomain",
        "average_number_of_hyphens_in_subdomain","having_special_characters_in_subdomain",
        "number_of_special_characters_in_subdomain","having_digits_in_subdomain",
        "number_of_digits_in_subdomain","having_repeated_digits_in_subdomain",
        "having_path","path_length","having_query","having_fragment",
        "having_anchor","entropy_of_url","entropy_of_domain"
    ]

    # Return as dict with ordered keys
    return {key: features[key] for key in feature_order}


In [ ]:
# Extract features for all URLs
print("Extracting features...")
features_list = []

for index, row in tqdm(combined_df.iterrows(), total=len(combined_df)):
    try:
        features = extract_detailed_features(row['url'], row['label'])
        features_list.append(features)
    except Exception as e:
        print(f"Error processing URL {row['url']}: {str(e)}")
        continue

# Create features DataFrame
features_df = pd.DataFrame(features_list)

print("Feature extraction completed!")
print("Final dataset shape:", features_df.shape)
print("Columns extracted:", features_df.columns.tolist())
print("Label distribution:\n", features_df['Type'].value_counts())

Extracting features...


100%|██████████| 59992/59992 [01:03<00:00, 950.29it/s] 


Feature extraction completed!
Final dataset shape: (59992, 43)
Columns extracted: ['Type', 'url_length', 'number_of_dots_in_url', 'having_repeated_digits_in_url', 'number_of_digits_in_url', 'number_of_special_char_in_url', 'number_of_hyphens_in_url', 'number_of_underline_in_url', 'number_of_slash_in_url', 'number_of_questionmark_in_url', 'number_of_equal_in_url', 'number_of_at_in_url', 'number_of_dollar_in_url', 'number_of_exclamation_in_url', 'number_of_hashtag_in_url', 'number_of_percent_in_url', 'domain_length', 'number_of_dots_in_domain', 'number_of_hyphens_in_domain', 'having_special_characters_in_domain', 'number_of_special_characters_in_domain', 'having_digits_in_domain', 'number_of_digits_in_domain', 'having_repeated_digits_in_domain', 'number_of_subdomains', 'having_dot_in_subdomain', 'having_hyphen_in_subdomain', 'average_subdomain_length', 'average_number_of_dots_in_subdomain', 'average_number_of_hyphens_in_subdomain', 'having_special_characters_in_subdomain', 'number_of_spe

In [ ]:
# Save the dataset
features_df.to_csv("detailed_url_features_dataset.csv", index=False)
print("Dataset saved as 'detailed_url_features_dataset.csv'")

# Download the dataset
files.download("detailed_url_features_dataset.csv")

# Display basic statistics
print("\nDataset Info:")
print(features_df.info())
print("\nBasic Statistics:")
print(features_df.describe())

Dataset saved as 'detailed_url_features_dataset.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59992 entries, 0 to 59991
Data columns (total 43 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Type                                       59992 non-null  int64  
 1   url_length                                 59992 non-null  int64  
 2   number_of_dots_in_url                      59992 non-null  int64  
 3   having_repeated_digits_in_url              59992 non-null  int64  
 4   number_of_digits_in_url                    59992 non-null  int64  
 5   number_of_special_char_in_url              59992 non-null  int64  
 6   number_of_hyphens_in_url                   59992 non-null  int64  
 7   number_of_underline_in_url                 59992 non-null  int64  
 8   number_of_slash_in_url                     59992 non-null  int64  
 9   number_of_questionmark_in_url              59992 non-null  int64  
 10  number_